# 01 Data Preprocessing and EDA


## Objective

完成数据预处理、派生特征构造、缺失值统计和探索性图表输出。本阶段新增按风险分组箱线图、类别变量风险率图，以及行为变量与结果变量关系图。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import CLASSIFICATION_TARGET, FIGURES_DIR, RANDOM_STATE, RESULTS_DIR
from data_utils import (
    describe_dataframe,
    ensure_project_dirs,
    load_raw_dataset,
    missing_value_summary,
    save_processed_dataset,
    validate_required_columns,
)
from feature_engineering import add_behavior_features
from visualization import (
    category_risk_rate_table,
    plot_behavior_outcome_scatter,
    plot_category_risk_rate,
    plot_correlation_heatmap,
    plot_eda_boxplots_by_risk,
    plot_missingness,
    plot_numeric_histograms,
    plot_target_distribution,
)

np.random.seed(RANDOM_STATE)
ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析


## Load, Validate, and Save Processed Dataset

这里仅做可复现的数据处理，不进行正式结论撰写。


In [2]:
df_raw = load_raw_dataset(download=True)
schema = validate_required_columns(df_raw)
df_processed = add_behavior_features(df_raw)
processed_path = save_processed_dataset(df_processed)
print(f"Processed dataset saved to: {processed_path}")
print(f"Processed shape: {df_processed.shape}")

display(schema)
df_processed.head()


Processed dataset saved to: C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\data\processed\digital_lifestyle_benchmark_2025_processed.csv
Processed shape: (3500, 31)


,column,present,dtype,missing_count
0,id,True,int64,0
1,age,True,int64,0
2,gender,True,str,0
3,region,True,str,0
4,income_level,True,str,0
5,education_level,True,str,0
6,daily_role,True,str,0
7,device_hours_per_day,True,float64,0
8,phone_unlocks,True,int64,0
9,notifications_per_day,True,int64,0


,id,age,gender,region,income_level,education_level,daily_role,device_hours_per_day,phone_unlocks,notifications_per_day,social_media_mins,study_mins,physical_activity_days,sleep_hours,sleep_quality,anxiety_score,depression_score,stress_level,happiness_score,focus_score,high_risk_flag,device_type,productivity_score,digital_dependence_score,social_media_hours,study_hours,notifications_per_device_hour,unlocks_per_device_hour,device_to_sleep_ratio,activity_sleep_interaction,social_to_study_ratio
0,1,40,Female,Asia,High,High School,Part-time/Shift,3.54,45,561,98,34,7.0,9.123800,3.353627,9.926651,5.0,6.593289,8.0,23.0,0,Android,70.000000,25.700000,1.633333,0.566667,158.474576,12.711864,0.387996,63.866600,2.800000
1,2,27,Male,Africa,Lower-Mid,Master,Full-time Employee,5.65,100,393,174,102,2.0,8.837517,2.908147,4.000000,4.0,4.126926,8.1,35.0,0,Laptop,64.000000,30.100000,2.900000,1.700000,69.557522,17.699115,0.639320,17.675034,1.689320
2,3,31,Male,North America,Lower-Mid,Bachelor,Full-time Employee,8.87,181,231,595,140,1.0,6.486743,2.889213,4.000000,8.0,1.429139,7.6,15.0,0,Android,65.299301,40.600000,9.916667,2.333333,26.042841,20.405862,1.367404,6.486743,4.219858
3,4,41,Female,Middle East,Low,Master,Caregiver/Home,4.05,94,268,18,121,4.0,7.600504,3.097488,7.093357,9.0,4.995512,7.8,28.0,1,Tablet,80.000000,36.684152,0.300000,2.016667,66.172840,23.209877,0.532859,30.402016,0.147541
4,5,26,Female,Europe,Lower-Mid,Bachelor,Full-time Employee,13.07,199,91,147,60,1.0,5.197962,2.786098,7.028125,15.0,9.448757,4.2,70.0,1,Android,65.299301,48.400000,2.450000,1.000000,6.962510,15.225708,2.514447,5.197962,2.409836


## Summary Tables

所有摘要表保存为 CSV，方便报告引用。新增的分组比较表和类别风险率表用于后续 EDA 小节。


In [3]:
descriptive_summary = describe_dataframe(df_processed)
missing_summary = missing_value_summary(df_processed)
target_distribution = (
    df_processed[CLASSIFICATION_TARGET]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis(CLASSIFICATION_TARGET)
    .reset_index(name="count")
)
target_distribution["ratio"] = target_distribution["count"] / target_distribution["count"].sum()

boxplot_columns = [
    "device_hours_per_day",
    "phone_unlocks",
    "notifications_per_day",
    "social_media_mins",
    "sleep_hours",
    "sleep_quality",
    "physical_activity_days",
    "digital_dependence_score",
]
category_columns = ["gender", "region", "income_level", "education_level", "daily_role", "device_type"]

group_comparison = (
    df_processed.groupby(CLASSIFICATION_TARGET)[boxplot_columns]
    .agg(["mean", "median", "std"])
    .reset_index()
)
group_comparison.columns = ["_".join([str(part) for part in col if str(part)]) for col in group_comparison.columns]
category_risk = category_risk_rate_table(df_processed, CLASSIFICATION_TARGET, category_columns)

schema.to_csv(RESULTS_DIR / "preprocessing_schema_check.csv", index=False)
descriptive_summary.to_csv(RESULTS_DIR / "eda_descriptive_summary.csv", index=False)
missing_summary.to_csv(RESULTS_DIR / "missing_value_summary.csv", index=False)
target_distribution.to_csv(RESULTS_DIR / "eda_high_risk_flag_distribution.csv", index=False)
group_comparison.to_csv(RESULTS_DIR / "eda_group_comparison_summary.csv", index=False)
category_risk.to_csv(RESULTS_DIR / "eda_category_risk_rate.csv", index=False)

display(target_distribution)
display(group_comparison)
display(category_risk.head(12))


,high_risk_flag,count,ratio
0,0,2795,0.798571
1,1,705,0.201429


,high_risk_flag,device_hours_per_day_mean,device_hours_per_day_median,device_hours_per_day_std,phone_unlocks_mean,phone_unlocks_median,phone_unlocks_std,notifications_per_day_mean,notifications_per_day_median,notifications_per_day_std,social_media_mins_mean,social_media_mins_median,social_media_mins_std,sleep_hours_mean,sleep_hours_median,sleep_hours_std,sleep_quality_mean,sleep_quality_median,sleep_quality_std,physical_activity_days_mean,physical_activity_days_median,physical_activity_days_std,digital_dependence_score_mean,digital_dependence_score_median,digital_dependence_score_std
0,0,6.773764,6.48,2.819610,137.064043,129.0,60.942994,331.884436,268.0,238.411718,157.221467,119.0,130.605111,7.445216,7.441119,1.205469,2.827614,2.955233,1.064518,3.345975,3.0,1.863107,34.520023,33.4,12.580636
1,1,9.474043,9.78,3.845289,186.924823,183.0,78.096286,347.933333,278.0,244.387581,167.852482,119.0,140.699877,6.498306,6.509528,1.345629,2.237994,2.063258,1.118303,3.365957,3.0,1.937072,45.263924,45.0,16.426778


,variable,category,count,risk_rate
0,gender,Female,1835,0.219074
1,gender,Male,1665,0.181982
2,region,Africa,578,0.193772
3,region,Asia,739,0.193505
4,region,Europe,797,0.200753
5,region,Middle East,339,0.224189
6,region,North America,622,0.196141
7,region,South America,425,0.216471
8,income_level,High,475,0.225263
9,income_level,Low,1139,0.193152


## Basic EDA Figures

这些图用于确认目标变量分布、缺失情况、数值变量分布和相关性结构。正式报告中只挑选与研究问题相关的图。


In [4]:
numeric_columns = df_processed.select_dtypes(include=[np.number]).columns.tolist()
plot_target_distribution(df_processed, CLASSIFICATION_TARGET, FIGURES_DIR / "eda_high_risk_flag_distribution.png")
plot_missingness(missing_summary, FIGURES_DIR / "eda_missing_value_rate.png")
plot_numeric_histograms(df_processed, numeric_columns, FIGURES_DIR / "eda_numeric_histograms.png")
plot_correlation_heatmap(df_processed, FIGURES_DIR / "eda_numeric_correlation_heatmap.png")
print("Saved basic EDA figures.")


Saved basic EDA figures.


## Risk Group Comparison Figures

按 `high_risk_flag` 分组的箱线图用于观察行为变量在高风险与非高风险样本之间是否存在分布差异。这里只提供可视化证据，是否能形成正式解释需要结合模型结果。


In [5]:
plot_eda_boxplots_by_risk(
    df_processed,
    CLASSIFICATION_TARGET,
    boxplot_columns,
    FIGURES_DIR / "eda_boxplots_by_risk.png",
)
print(FIGURES_DIR / "eda_boxplots_by_risk.png")


C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\figures\eda_boxplots_by_risk.png


## Category Risk Rate Figure

类别变量风险率图只用于描述不同背景组的样本风险率，不代表因果关系。后续建模仍会严格区分输入特征与结果变量。


In [6]:
plot_category_risk_rate(category_risk, FIGURES_DIR / "eda_category_risk_rate.png")
print(FIGURES_DIR / "eda_category_risk_rate.png")


C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\figures\eda_category_risk_rate.png


## Behavior and Outcome Relationship Figure

散点图和分箱均值图用于辅助判断数字行为变量与结果变量之间是否存在可建模关系。正式报告中需要避免把相关性描述成因果结论。


In [7]:
plot_behavior_outcome_scatter(df_processed, FIGURES_DIR / "eda_behavior_outcome_scatter.png")
print(FIGURES_DIR / "eda_behavior_outcome_scatter.png")


C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\figures\eda_behavior_outcome_scatter.png
